# EXP-017 CPU 단계 질의당 시간

운영 구성의 CPU 부분(질의 임베딩, 1차 검색, 재정렬 후보 20개 청크 준비)을 로컬 PC에서 질의 하나씩 잰다. GPU 재정렬 시간(`rag.rerank mode=latency`)과 같은 질의 100개를 쓴다(`docs/experiments/EXP-017.md` 질의당 시간 측정).

In [1]:
import gc
import json
import os
import time
from itertools import groupby
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from rag.conditions import build_extractor
from rag.generate import ChunkStore, best_chunks
from rag.index import encoder, load_cfg, load_model
from rag.rerank import read_pairs
from rag.retrieve import Retriever, chunks_by_doc

C:\workspace\meet-rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
ROOT = Path.cwd().parent
os.chdir(ROOT)  # 설정의 상대 경로를 저장소 기준으로 푼다
cfg = load_cfg(["+exp=exp017", "index.scope=full"])
LAT = cfg.retriever.rerank.latency
DEPTH = cfg.retriever.rerank.depth
OUT = ROOT / "data/rerank/latency-summary_q-full-ms256-k20-cpu.json"
LAT, DEPTH, cfg.paths.rerank_pairs

({'n_queries': 100, 'seed': 20260929},
 20,
 'data/rerank/summary_q-full-ms256-k20.pairs.jsonl')

## 표본

`rag.rerank`의 `latency`와 같은 방식으로 고른다: 후보 쌍 파일의 질의 순서에서 seed로 101개를 뽑고, 첫 질의는 준비 실행으로 뺀다.

In [3]:
pairs = read_pairs(ROOT / cfg.paths.rerank_pairs)
by_qid = [list(g) for _, g in groupby(pairs, key=lambda p: p["qid"])]
rng = np.random.default_rng(LAT.seed)
picked = [by_qid[i] for i in rng.choice(len(by_qid), LAT.n_queries + 1, replace=False)]
queries = [(g[0]["qid"], g[0]["query"]) for g in picked]
del pairs, by_qid
len(queries), [q for q, _ in queries[:3]]

(101, ['048563-0008', '049487-0004', '035647-0001'])

## 질의 임베딩 (캐시 없음)

로컬 PC 메모리(8GB)에 임베딩 모델과 전체 인덱스를 함께 올리면 커널이 죽어(첫 실행) 단계를 나눠 잰다. 모델로 질의를 하나씩 임베딩해 시간을 재고 모델을 내린 뒤, 그 임베딩으로 검색 단계를 잰다. 단계는 순서대로 이어지므로 질의당 합계는 한 프로세스에서 잰 것과 같다(추론). 모델 로딩 시간은 콜드 스타트에 넣는다.

In [4]:
t0 = time.perf_counter()
model = load_model(cfg, cfg.embedding.query_max_tokens)
encode = encoder(model, cfg, cfg.embedding.get("query_prompt"))
cold_model = time.perf_counter() - t0
embs, embed_t = [], []
for _, text in queries:
    t0 = time.perf_counter()
    embs.append(encode([text])[0])
    embed_t.append(time.perf_counter() - t0)
del model, encode
gc.collect()
print(f"모델 로딩 {cold_model:.1f}초, torch 스레드 {torch.get_num_threads()}, "
      f"임베딩 평균 {np.mean(embed_t[1:]):.3f}초")

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 2102.60it/s]


모델 로딩 4.9초, torch 스레드 4, 임베딩 평균 0.296초


## 인덱스 준비 (콜드 스타트)

인덱스·BM25·조건 색인·청크 위치를 한 번 읽는다. 운영에서는 시작할 때 한 번 드는 시간이다.

In [5]:
t0 = time.perf_counter()
ret = Retriever(cfg)
extractor = build_extractor(cfg)
doc_chunks = chunks_by_doc(ret.dense_ids)
store = ChunkStore(ret.dirs)
cold_index = time.perf_counter() - t0
cold = cold_model + cold_index
print(f"인덱스 로딩 {cold_index:.1f}초, 콜드 스타트 합계 {cold:.1f}초, 청크 {len(ret.dense_ids):,}개")

인덱스 로딩 48.6초, 콜드 스타트 합계 53.5초, 청크 338,684개


## 질의당 CPU 단계

단계마다 잰다: 질의 임베딩(위 셀), 1차 검색(dense 내적, BM25, 조건 추출, RRF 결합), 후보 준비(RRF 상위 20개 문서의 dense 최고점 청크 본문 읽기). 재정렬은 필터 전 RRF 순위에 하므로 1차 검색은 `candidates`까지다.

In [6]:
rows = []
for (qid, text), emb, te in zip(queries, embs, embed_t, strict=True):
    t1 = time.perf_counter()
    row = ret.chunk_emb @ emb
    tokens = ret.query_tokens([text])[0]
    docs, matched, _ = ret.candidates(row, tokens, extractor.extract(text))
    t2 = time.perf_counter()
    texts = [store.get(i)["text"] for i in best_chunks(row, doc_chunks, docs[:DEPTH])]
    t3 = time.perf_counter()
    rows.append({"qid": qid, "embed": te, "retrieve": t2 - t1, "prepare": t3 - t2,
                 "n_texts": len(texts)})
lat = pd.DataFrame(rows[1:])  # 첫 질의는 준비 실행
lat["cpu_total"] = lat[["embed", "retrieve", "prepare"]].sum(axis=1)
lat.shape, lat["n_texts"].value_counts().to_dict()

((100, 6), {20: 100})

In [9]:
stats = lat[["embed", "retrieve", "prepare", "cpu_total"]].describe(percentiles=[0.5, 0.95]).T
stats[["mean", "50%", "95%", "max"]].round(3)

,mean,50%,95%,max
embed,0.296,0.261,0.477,0.531
retrieve,0.104,0.088,0.135,0.139
prepare,0.003,0.003,0.003,0.007
cpu_total,0.403,0.369,0.608,0.644


In [11]:
result = {"threads": torch.get_num_threads(), "n_queries": len(lat), "cold_start": cold,
          **{f"{c}_{s}": float(v) for c in ["embed", "retrieve", "prepare", "cpu_total"]
             for s, v in [("mean", lat[c].mean()), ("p50", lat[c].quantile(0.5)),
                          ("p95", lat[c].quantile(0.95)), ("max", lat[c].max())]}}
OUT.write_text(json.dumps(result, indent=2) + "\n", encoding="utf-8")
result["cpu_total_mean"], result["cpu_total_p95"]

(0.402821690995479, 0.6079813800170086)